# 1. Setup

In [ ]:
import logging
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd


def _setup_project_path() -> Path:
    """Finds the project root folder and appends <root>/src to sys.path."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path

    raise RuntimeError("Could not find the project path. Expected a folder named 'src/byeias'.")

# Add the source path BEFORE importing custom modules
SRC_PATH = _setup_project_path()

# Custom package imports ('# noqa: E402' suppresses linter warnings for late imports)
from byeias.backend.classification.model_bias import BiasDetectionPipeline  # noqa: E402
from byeias.backend.config_loader import get_backend_config  # noqa: E402
from byeias.backend.extraction.text_extracter import PDFTextExtractor  # noqa: E402
from byeias.backend.llm_explanation.llm_communicator import LLMCommunicator  # noqa: E402

In [1]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

NameError: name 'logging' is not defined

# 2. Generate Testdata

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

# 3. PDF Extraction

In [ ]:
def test_pdf_extraction(pdf_path_str: str) -> None:
    """Tests the PDFTextExtractor."""
    logger.info("--- Testing PDF Extraction ---")
    pdf_path = Path(pdf_path_str)
    
    if not pdf_path.exists():
        logger.error(f"File not found: {pdf_path.resolve()}")
        return

    extractor = PDFTextExtractor(language="german")
    sentences = extractor.extract_sentences(str(pdf_path))
    
    logger.info(f"Extracted sentences: {len(sentences)}")
    for i, sentence in enumerate(sentences[:5], start=1):
        print(f"{i}: {sentence}")

# 4. Classification (BERT)

In [ ]:
def test_model_inference(use_loaded_weights: bool = True) -> None:
    """Tests model prediction (inference), optionally with pre-loaded weights."""
    logger.info("--- Testing Model Inference ---")
    config = get_backend_config()
    pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)

    if use_loaded_weights:
        weights_path = Path(config.classification.best_model_path)
        if weights_path.exists():
            logger.info(f"Loading existing weights from: {weights_path}")
            try:
                pipeline.load_model(str(weights_path))
                logger.info("Model loaded successfully!")
            except Exception as e:
                logger.error(f"Critical error loading weights: {e}", exc_info=True)
                return
        else:
            logger.warning(f"Weights file not found at {weights_path}. Using base model instead.")

    test_samples = build_test_samples()
    test_contexts = [s["context"] for s in test_samples]
    test_texts = [s["text"] for s in test_samples]

    logger.info("Starting predictions...")
    predictions = pipeline.predict(context_texts=test_contexts, target_texts=test_texts)

    print("\n--- INFERENCE RESULTS ---")
    for idx, res in enumerate(predictions, start=1):
        sexism_flag = "YES" if res['sexism_prediction'] == 1 else "NO"
        racism_flag = "YES" if res['racism_prediction'] == 1 else "NO"
        
        print(f"Sample {idx}:")
        print(f"  Context: '{res['context']}'")
        print(f"  Text:    '{res['text']}'")
        print(f"  -> Sexism: {sexism_flag}")
        print(f"  -> Racism: {racism_flag}\n")

# 5. LLM Communication (Mistral)

In [ ]:
def test_llm_communication() -> None:
    """Tests communication with the LLM for generating bias explanations."""
    logger.info("--- Testing LLM Communication ---")
    context_before = "Im Klassenzimmer ging es um Mathematik."
    flagged_sentence = "Mädchen sind von Natur aus schlechter in Mathe."
    context_after = "Die Lehrerin erklärte die nächste Aufgabe."

    llm = LLMCommunicator()
    try:
        result = llm.explain_bias(context_before, flagged_sentence, context_after)
        print("\nLLM Result:")
        print(result)
    except Exception as e:
        logger.error(f"Error during LLM communication: {e}", exc_info=True)

# 6. Full Pipeline

In [ ]:
def test_full_pipeline(input_text: str) -> List[Dict[str, Any]]:
    """Tests the complete processing pipeline: Sentence splitting -> Inference -> LLM Explanation."""
    logger.info("--- Testing Complete Pipeline ---")
    
    # Split text into sentences (simple logic based on punctuation)
    raw_sentences = re.split(r'(?<=[.!?])\s+', input_text.strip())
    sentences = [s.strip() for s in raw_sentences if s.strip()]

    logger.info(f"Input text split into {len(sentences)} sentences.")
    if not sentences:
        logger.warning("No sentences found to process.")
        return []

    # Initialize components
    config = get_backend_config()
    pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)
    
    # Optional: Try to load pre-trained weights
    try:
        pipeline.load_model(config.classification.best_model_path)
    except Exception as e:
        logger.warning(f"Could not load weights, continuing with base model. Reason: {e}")

    llm = LLMCommunicator()

    # Prepare context lists
    context_texts, target_texts = [], []
    contexts_before, contexts_after = [], []

    for i in range(len(sentences)):
        target = sentences[i]
        before = sentences[i-1] if i > 0 else ""
        after = sentences[i+1] if i + 1 < len(sentences) else ""
        
        combined_context = f"{before} {after}".strip()
        
        target_texts.append(target)
        context_texts.append(combined_context)
        contexts_before.append(before)
        contexts_after.append(after)

    # Generate predictions
    logger.info("Starting classification for all sentences...")
    inference_results = pipeline.predict(context_texts=context_texts, target_texts=target_texts)

    explanations = []
    
    # Evaluate results and fetch LLM explanations for flagged sentences
    for i, result in enumerate(inference_results):
        is_sexist = result["sexism_prediction"] == 1
        is_racist = result["racism_prediction"] == 1
        
        if is_sexist or is_racist:
            logger.info(f"[BIAS FOUND] in sentence {i+1}: '{result['text']}'")
            bias_type = "Sexism" if is_sexist else "Racism"
            
            try:
                explanation = llm.explain_bias(
                    context_before=contexts_before[i],
                    flagged_sentence=result["text"],
                    context_after=contexts_after[i],
                )
            except Exception as e:
                logger.error(f"LLM Error at sentence {i+1}: {e}")
                explanation = "Explanation could not be generated."

            explanations.append({
                "sentence_index": i + 1,
                "flagged_sentence": result["text"],
                "bias_type": bias_type,
                "llm_explanation": explanation
            })

    return explanations

# 7. Tests

In [ ]:

# 1. PDF Extraction
test_pdf_extraction("../../../../data/....pdf")
    
# 3. Model Inference (with or without saved weights)
test_model_inference(use_loaded_weights=False) 
    
# 4. LLM Communication
test_llm_communication()
    
# 5. Full Pipeline
sample_text = (
    "The Topic is about payment. Girls get paid less. "
    "The teacher explained the next task. Everyone was listening."
)
results = test_full_pipeline(sample_text)
if results:
    print("\n" + "="*50)
    print("FINAL SUMMARY")
    print("="*50)
    for res in results:
        print(f"Sentence {res['sentence_index']} ({res['bias_type']}): {res['flagged_sentence']}")
        print(f"Explanation:\n{res['llm_explanation']}\n")
        print("-" * 50)
else:
    print("\nNo bias found in the text or processing failed.")

# PDF Extraction

In [ ]:
# Teste PDF Extraction
import sys
from pathlib import Path
def _add_src_to_path():
    from pathlib import Path
    import sys
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path
    raise RuntimeError("Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'.")
_add_src_to_path()
from byeias.backend.extraction.text_extracter import PDFTextExtractor

# Beispiel-PDF (muss existieren)
pdf_path = "../../../../data/Businessplan_Halo_AktuellerStand.pdf"

extractor = PDFTextExtractor(language="german")
# Prüfe, ob die PDF-Datei existiert, bevor extrahiert wird

pdf_file = Path(pdf_path)
if not pdf_file.exists():
    print(f"Datei nicht gefunden: {pdf_file.resolve()}")
    sentences = []
else:
    sentences = extractor.extract_sentences(pdf_path)
    
    print(f"Extrahierte Sätze: {len(sentences)}")
for i, satz in enumerate(sentences[:5], 1):
    print(f"{i}: {satz}")

# Classification

In [ ]:
import logging
from pathlib import Path

import pandas as pd

def _add_src_to_path():
    """Findet den Projektroot und fügt <root>/src zu sys.path hinzu."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path

    raise RuntimeError(
        "Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'."
    )

SRC_PATH = _add_src_to_path()
print(f"Nutze Python-Pfad: {SRC_PATH}")

# Importiere BiasDetectionPipeline aus dem richtigen Modulpfad
from byeias.backend.classification.model_bias import BiasDetectionPipeline  # noqa: E402

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
import pandas as pd

def build_tiny_datasets():
    """Very small datasets to quickly test the pipeline functionality."""
    
    # Sexism Training Dataset
    train_df_sexism = pd.DataFrame(
        {
            "context": [
                "The team meeting was about leadership roles.",
                "New projects were discussed in the office.",
                "The application was about qualifications.",
                "Grades were distributed at school.",
                "A discussion about household chores."
            ],
            "text": [
                "Women are too emotional for leadership positions.",
                "Everyone collaborated constructively.",
                "Men are better suited for technical jobs.",
                "The class learned together fairly.",
                "Taking care of children is a woman's job."
            ],
            "sexism_label": [1, 0, 1, 0, 1], # Adjusted to 5 labels
        }
    )

    # Racism Training Dataset
    train_df_racism = pd.DataFrame(
        {
            "context": [
                "During a discussion about migration.",
                "A festival took place in the city park.",
                "The lesson was about cultural diversity.",
                "Many nations were represented at the sports event.",
                "Talking about neighborhood safety."
            ],
            "text": [
                "People from this country are all criminals.",
                "All families celebrated together.",
                "Foreigners never want to integrate.",
                "The tournament was respectful and fair.",
                "They don't belong in our neighborhood."
            ],
            "racism_label": [1, 0, 1, 0, 1], # Adjusted to 5 labels
        }
    )

    # Sexism Validation Dataset
    val_df_sexism = pd.DataFrame(
        {
            "context": [
                "Performance was evaluated in the job interview.",
                "Everyone in the club planned the event together.",
            ],
            "text": [
                "Women should stay out of the executive suite.",
                "Tasks were distributed regardless of gender.",
            ],
            "sexism_label": [1, 0],
        }
    )

    # Racism Validation Dataset
    val_df_racism = pd.DataFrame(
        {
            "context": [
                "The canteen conversation was about origins.",
                "Examples from many countries were shown in the seminar.",
            ],
            "text": [
                "This ethnicity is fundamentally lazy.",
                "Different backgrounds enrich the discussion.",
            ],
            "racism_label": [1, 0],
        }
    )

    return train_df_sexism, train_df_racism, val_df_sexism, val_df_racism

In [ ]:
def build_test_samples(n=5):
    """Generates n test samples for prediction."""
    samples = [
        {
            "context": "The topic in the classroom was mathematics.",
            "text": "Girls are naturally worse at math.",
        },
        {
            "context": "A new team was formed in the company.",
            "text": "The collaboration was professional and respectful.",
        },
        {
            "context": "In a debate about immigration.",
            "text": "All migrants are a burden.",
        },
        {
            "context": "Many cultures were represented at the city festival.",
            "text": "The festival was open and inclusive.",
        },
        {
            "context": "Roles were assigned in the project meeting.",
            "text": "Women should rather take on supporting tasks.",
        },
    ]
    return samples[:n]

In [ ]:
n_predictions = 5
train_df_sexism, train_df_racism, val_df_sexism, val_df_racism = build_tiny_datasets()
test_samples = build_test_samples(n=n_predictions)
test_contexts = [sample["context"] for sample in test_samples]
test_texts = [sample["text"] for sample in test_samples]

print(f"Train Sexism: {len(train_df_sexism)} | Train Racism: {len(train_df_racism)}")
print(f"Val Sexism: {len(val_df_sexism)} | Val Racism: {len(val_df_racism)}")
print(f"Inference Beispiele: {len(test_texts)}")

In [ ]:
logger.info("Initialisiere Pipeline...")
pipeline = BiasDetectionPipeline(model_name="microsoft/deberta-v3-small")
print("Pipeline initialisiert")

In [ ]:
logger.info("Starte kurzes Training...")
pipeline.train(
    train_df_sexism=train_df_sexism,
    train_df_racism=train_df_racism,
    val_df_sexism=val_df_sexism,
    val_df_racism=val_df_racism,
    epochs=1,
    batch_size=2,
    lr=2e-5,
    )
print("Training abgeschlossen")

In [ ]:
# --- VORHERSAGE (Inference) ---
logger.info("Starte Vorhersage...")
predictions = pipeline.predict(context_texts=test_contexts, target_texts=test_texts)
print(f"Vorhersagen erstellt: {len(predictions)}")

In [ ]:
print("\n--- INFERENCE ERGEBNISSE ---")
for idx, res in enumerate(predictions, start=1):
    print(f"Beispiel {idx}:")
    print(f"Kontext: '{res['context']}'")
    print(f"Text:    '{res['text']}'")
    print(f" -> Sexismus: {'JA' if res['sexism_prediction'] == 1 else 'NEIN'}")
    print(f" -> Rassismus: {'JA' if res['racism_prediction'] == 1 else 'NEIN'}\n")

# Clasification 2

In [ ]:
# --- CLASSIFICATION (Laden statt Trainieren) ---
import logging
import os
from pathlib import Path
import pandas as pd

# (Deine _add_src_to_path Funktion hier lassen...)
SRC_PATH = _add_src_to_path()
from byeias.backend.classification.model_bias import BiasDetectionPipeline
from byeias.backend.config_loader import get_backend_config

# Lade Konfiguration, um den Pfad zu den Gewichten zu erhalten
config = get_backend_config()
weights_path = config.classification.best_model_path

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialisiere die Pipeline
pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)

# --- MODELL LADEN MIT FEHLER-ANALYSE ---
if Path(weights_path).exists():
    logger.info(f"Lade existierende Gewichte von: {weights_path}")
    try:
        # Hier bleibt das Skript aktuell stecken
        pipeline.load_model(weights_path)
        print("Modell erfolgreich geladen!")
    except Exception as e:
        print("\n" + "="*50)
        print(f"KRITISCHER FEHLER BEIM LADEN DER GEWICHTE:")
        print(e)
        print("="*50 + "\n")
else:
    print(f"WARNUNG: Gewichtsdatei nicht gefunden unter {weights_path}")
# Test-Daten vorbereiten
test_samples = build_test_samples(n=5)
test_contexts = [sample["context"] for sample in test_samples]
test_texts = [sample["text"] for sample in test_samples]

# --- VORHERSAGE (Inference) ---
logger.info("Starte Vorhersage...")
predictions = pipeline.predict(context_texts=test_contexts, target_texts=test_texts)

print("\n--- INFERENCE ERGEBNISSE ---")
for idx, res in enumerate(predictions, start=1):
    print(f"Beispiel {idx}:")
    print(f"Kontext: '{res['context']}'")
    print(f"Text:    '{res['text']}'")
    print(f" -> Sexismus: {'JA' if res['sexism_prediction'] == 1 else 'NEIN'}")
    print(f" -> Rassismus: {'JA' if res['racism_prediction'] == 1 else 'NEIN'}\n")

# LLM Communication

In [ ]:
# Teste LLM Communication
from pathlib import Path

def _add_src_to_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path
    raise RuntimeError("Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'.")

_add_src_to_path()
from byeias.backend.llm_explanation.llm_communicator import LLMCommunicator

# Beispiel-Kontext
context_before = "Im Klassenzimmer ging es um Mathematik."
flagged_sentence = "Mädchen sind von Natur aus schlechter in Mathe."
context_after = "Die Lehrerin erklärte die nächste Aufgabe."

llm = LLMCommunicator()
result = llm.explain_bias(context_before, flagged_sentence, context_after)
print("LLM-Ergebnis:")
print(result)

# Complete Pipeline

In [ ]:
import re

def process_data(input_text):
    raw_sentences = re.split(r'(?<=[.!?])\s+', input_text.strip())
    sentences = [s.strip() for s in raw_sentences if s.strip()]

    print(f"Eingabetext in {len(sentences)} Sätze aufgeteilt.")

    if not sentences:
        print("Keine Sätze zum Verarbeiten gefunden.")
        return []

    pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)
    llm = LLMCommunicator()

    context_texts = []
    target_texts = []
    contexts_before = []
    contexts_after = []

    for i in range(len(sentences)):
        target = sentences[i]
        
        before = sentences[i-1] if i > 0 else ""
        
        after = sentences[i+1] if i + 1 < len(sentences) else ""
        
        combined_context = f"{before} {after}".strip()
        
        target_texts.append(target)
        context_texts.append(combined_context)
        
        contexts_before.append(before)
        contexts_after.append(after)

    print("Starte Vorhersage für alle Sätze...")
    inference_results = pipeline.predict(
        context_texts=context_texts, 
        target_texts=target_texts
    )

    print("Inferenz-Ergebnisse:", inference_results)

    explanations = []

    for i, result in enumerate(inference_results):
        if result["sexism_prediction"] == 1 or result["racism_prediction"] == 1:
            print(f"\n[BIAS GEFUNDEN] in Satz {i+1}: '{result['text']}'")
            
            explanation = llm.explain_bias(
                context_before=contexts_before[i],
                flagged_sentence=result["text"],
                context_after=contexts_after[i],
            )
            
            explanations.append({
                "satz_index": i + 1,
                "geflaggter_satz": result["text"],
                "bias_typ": "Sexismus" if result["sexism_prediction"] == 1 else "Rassismus",
                "llm_erklaerung": explanation
            })

    return explanations

# --- TEST ---
text = "The Topic is about payment. Girls get paid less. The teacher explained the next task. Everyone was listening."
ergebnisse = process_data(text)

print("\n--- FINALE ZUSAMMENFASSUNG ---")
for erg in ergebnisse:
    print(f"Satz {erg['satz_index']} ({erg['bias_typ']}): {erg['geflaggter_satz']}")
    print(f"Erklärung: {erg['llm_erklaerung']}\n")